In [1]:
import numpy as np
import pandas as pd
import openpyxl
import os

In [2]:
path_produ = "lisempporconcespley.xlsx"
path_decreto_uno = "68-25 arsi - 68-25 arsi.csv"
path_decreto_dos = "68-25 arsi higiene.csv"

Lectura del excel de productividades.
Me importan tener las demás columnas? la fecha hay que chequear algo?
Además, ordenamos por legajos para hacer más eficiente la búsqueda.

In [3]:
df_prod = pd.read_excel(path_produ)
df_prod.sort_values(by = "Legajo")

,Legajo,Cargo,Apellido y Nombre,Unnamed: 3,Leyenda,Inicio,Unnamed: 6,Fin,Cantidad,Base,Importe,Porcentaje,Indicativo
1387,11529,1900-04-16,CORREA URQUIZA DOLORES,NaN,DTO. 68/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,87612.00,00:00:00,ADICIONAL
1388,11529,1900-04-16,CORREA URQUIZA DOLORES,NaN,DTO. 68/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,223500.00,00:00:00,ADICIONAL
2080,11894,1900-04-12,TORRES ROSA MARGARITA DEL VALLE,NaN,DTO. 51/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,10000.00,00:00:00,ADICIONAL
395,12118,1900-04-18,MEDINA LUIS ENRIQUE,NaN,DTO. 181/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,64152.00,00:00:00,ADICIONAL
2633,12348,1900-04-12,LUJAN RICARDO ALBERTO,NaN,DTO. 203/25 (EVENTOS Y GUARDIAS HIGIENE),1/12/2025,NaN,31/12/2025,1900-01-04 00:00:00,2018-03-17 16:48:00,172706.80,00:00:00,ADICIONAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...
56,77319,1900-01-01,PACHECOS ROSAS ANGHELINA GERALDINE,NaN,DTO. 222/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,68153.18,00:00:00,ADICIONAL
355,77322,1900-01-01,ALLER NATALIA,NaN,DTO. 219/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,53460.00,00:00:00,ADICIONAL
354,77322,1900-01-01,ALLER NATALIA,NaN,DTO. 222/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,94458.88,00:00:00,ADICIONAL
363,77325,1900-01-01,IARLORI CAMILA,NaN,DTO. 222/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,94458.88,00:00:00,ADICIONAL


Lectura de los archivos por decreto. Pasamos la columna de legajo a tipo int64, de esta manera tipa con la columna legajo de df_prod. Además, ordenamos según legajo para hacer más eficiente la búsqueda.

In [101]:
df_decreto = pd.read_csv(path_decreto,header=None)
df_decreto.columns = ["Legajo", "Nula", "Nula2", "Importe"]
df_decreto = df_decreto.dropna()
df_decreto["Legajo"] = df_decreto["Legajo"].astype('Int64')
df_decreto.sort_values(by = "Legajo")


,Legajo,Nula,Nula2,Importe
5,14820,0.0,0.0,524731.90
4,17714,0.0,0.0,524731.90
2,18333,0.0,0.0,203876.39
0,55339,0.0,0.0,524731.90
1,66515,0.0,0.0,203876.39
3,68048,0.0,0.0,524731.90


Obtengo el decreto del archivo en el cuál estamos trabajando. Y veo si tipa con el tipo de df_prod.  

In [77]:
decreto = path_decreto.split(".")[0]
decreto = decreto.split("-")[0] + "/" +decreto.split("-")[1]
decreto

'47/25'

Limpiamos los nombres de la columna decretos, para que nos quede de la forma num/año. (IMPORTANTE) correr solo una vez.

In [71]:
cant_prod = df_prod.shape[0]

for i in range(cant_prod):

    leyenda = df_prod.iloc[i]["Leyenda"]
    decreto_prod = leyenda.split(" ")[1]
    df_prod.loc[i,"Leyenda"] = decreto_prod

df_prod["Leyenda"] = df_prod["Leyenda"].astype('str')


In [78]:
df_prod_dec = df_prod[df_prod["Leyenda"] == decreto]

In [79]:
df_prod_dec

,Legajo,Cargo,Apellido y Nombre,Unnamed: 3,Leyenda,Inicio,Unnamed: 6,Fin,Cantidad,Base,Importe,Porcentaje,Indicativo
30,17714,1900-04-16,CONSOLI MARIA JOSE,NaN,47/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,524731.90,00:00:00,ADICIONAL
32,18333,1900-04-19,GARCIA STEFANI VERONICA CARINA,NaN,47/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,203876.39,00:00:00,ADICIONAL
34,66515,1900-04-11,GIANNONE GABRIELA,NaN,47/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,203876.39,00:00:00,ADICIONAL
36,68048,1900-04-11,TORRES TORANZO LUCIANO,NaN,47/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,524731.90,00:00:00,ADICIONAL
38,14820,1900-04-15,MONACO ROSANA ELIZABETH,NaN,47/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,524731.90,00:00:00,ADICIONAL
40,55339,1900-04-19,MORELLO NATALIA SOLEDAD,NaN,47/25,1/12/2025,NaN,31/12/2025,00:00:00,00:00:00,524731.90,00:00:00,ADICIONAL


In [84]:
def binary_search(df, column_name, x):
    """
    Búsqueda binaria en una columna de un DataFrame.
    
    Args:
        df: DataFrame de pandas
        column_name: nombre de la columna donde buscar
        x: valor a buscar
    
    Returns:
        índice (posición) donde se encuentra el valor, o -1 si no existe
    """
    low = 0
    high = len(df) - 1

    while low <= high:
        mid = low + (high - low) // 2

        if df.iloc[mid][column_name] < x:
            low = mid + 1
        elif df.iloc[mid][column_name] > x:
            high = mid - 1
        else:
            return mid
    return -1

In [85]:
binary_search(df_prod_dec,"Legajo",66515)

2

In [104]:
df_diferencias = pd.DataFrame(columns=["Legajo", "Decreto", "Importe"])

In [109]:
def comparar(df_prod_dec: pd.DataFrame, df_dec: pd.DataFrame) -> None:

    """
    Toma el df de productividades filtrado por decreto y se fija si encuentra o no el monto correspondiente a cada legajo

    """

    cant_prod_dec = df_prod_dec.shape[0]
    cant_dec = df_dec.shape[0]
    decreto = df_prod_dec["Leyenda"].unique()[0] #Nombre del decreto

    for i in range(cant_prod_dec):

        legajo = df_prod_dec.iloc[i]["Legajo"]
        importe = df_prod_dec.iloc[i]["Importe"]

        #Busco si existe la fila en el dataFrame correspondiente al decreto

        existe = False

        for j in range(cant_dec):

            legajo_dec = df_dec.iloc[j]["Legajo"]
            importe_dec = df_dec.iloc[j]["Importe"]

            if importe_dec == importe and legajo_dec == legajo:
                
                existe = True

        if existe == False: #Agregar a un dataFrame global que sea el de diferencias

            df_diferencias.loc[len(df_diferencias)] = [legajo, decreto, importe]

In [107]:
df_decreto

,Legajo,Nula,Nula2,Importe
0,55339,0.0,0.0,524731.90
1,66515,0.0,0.0,203876.39
2,18333,0.0,0.0,203876.39
3,68048,0.0,0.0,524731.90
4,17714,0.0,0.0,524731.90
5,14820,0.0,0.0,524731.90


In [106]:
comparar(df_prod_dec, df_decreto)

df_diferencias

,Legajo,Decreto,Importe
